# Square hard-wall purification: one trajectory each at 30×30 and 40×40

60 physical cycles; alpha1=1, alpha2=30, nshell=1, hard support-truncated walls,
global maximally mixed initialization, full-system measurements, raster-y,
perfect correction, complex128. No postselection or state clipping.
Use an A100 40-GB-class runtime. Each size is independently resumable.

At every cycle including zero, save total mixed-state von Neumann entropy,
cell-resolved entropy contour, occupations and gap diagnostics. For C=(G+I)/2,
S=sum h(nu) and s_i=sum_j |U_ij|² h(nu_j), summed over the two orbitals per cell.
This is purification entropy, not the entanglement of a spatial subsystem.
Natural logs; the 1e-12 entropy endpoint regulator matches the preceding campaign.
The state itself is never clipped. A fully capped spectrum has an undefined gap.

Five-cycle rolling covariance/RNG/observer checkpoints limit restart loss.
Only one writer may use this output folder. No covariance or eigenvector history.
No sample error bars are meaningful with one trajectory per size.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

REPORT_ONLY = False
MAX_NEW_TASKS = None  # 1 runs only the first unfinished size
# Scientific edits require a new sampling_revision/output folder.
CONFIG = {'sampling_revision': 'square_full_measurement_purification_l30-40_s1_t60_v1',
 'root_seed': 2026092929,
 'sizes': [30, 40],
 'samples': 1,
 'cycles': 60,
 'segment_cycles': 5,
 'DW': True,
 'dw_truncation': True,
 'meas_slab_only': False,
 'triv_region_local_mode': False,
 'alpha_1': 1.0,
 'alpha_2': 30.0,
 'nshell': 1,
 'trial_orbitals': 'X',
 'filling_fraction': 0.5,
 'init_mode': 'maxmix',
 'n_a': 0.5,
 'sequence': 'raster_y',
 'perfect_correction': True,
 'postselect': False,
 'state_representation': 'covariance',
 'dtype': 'complex128',
 'device': 'cuda:0',
 'backend': 'local',
 'covariance_spectral_clip': False,
 'occupation_tolerance': 1e-09,
 'entropy_eps': 1e-12,
 'initial_purity_tolerance': 0.50000001}
BUNDLE_DRIVE_DIR = Path('/content/drive/MyDrive/final_production_new_designs/29_square_purification_contour_pilot')
LOCAL_BUNDLE_DIR = Path('/content/29_square_purification_contour_pilot')
SCRATCH_ROOT = Path('/content/square_contour_scratch')
OUTPUT_ROOT = Path('/content/drive/MyDrive/classA_final_production_outputs') / CONFIG['sampling_revision']


## Stage locally and run/resume

Visible outer task and inner cycle progress includes eigensolving. Completed result pairs are verified and skipped. Active tasks resume from their own five-cycle checkpoints.

In [ ]:
import codecs
import json
import shutil
import subprocess
import sys

LOCAL_BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
for relative in ['run_campaign.py', 'contour_observer.py', 'io_utils.py', 'src/classA_U1FGTN_gpu.py', 'src/occupied_frame_gpu.py']:
    destination = LOCAL_BUNDLE_DIR / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(BUNDLE_DRIVE_DIR / relative, destination)
config_path = LOCAL_BUNDLE_DIR / 'resolved_config.json'
config_path.write_text(json.dumps(CONFIG, indent=2))
command = [sys.executable, '-u', str(LOCAL_BUNDLE_DIR/'run_campaign.py'),
           '--config', str(config_path), '--output-root', str(OUTPUT_ROOT), '--scratch-root', str(SCRATCH_ROOT)]
if REPORT_ONLY: command.append('--report-only')
if MAX_NEW_TASKS is not None: command.extend(['--max-new-tasks', str(MAX_NEW_TASKS)])
print('[launch]', ' '.join(command), flush=True)
decoder = codecs.getincrementaldecoder('utf-8')('replace')
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0) as process:
    try:
        while True:
            chunk = process.stdout.read(4096)
            if not chunk: break
            sys.stdout.write(decoder.decode(chunk)); sys.stdout.flush()
        sys.stdout.write(decoder.decode(b'', final=True)); sys.stdout.flush()
        code = process.wait()
        if code: raise subprocess.CalledProcessError(code, command)
    except BaseException:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
        raise


## Quick view of completed trajectories

No averaging or uncertainty bands; these are individual trajectories. Cycle zero is omitted only from logarithmic time plots.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, str(LOCAL_BUNDLE_DIR))
import run_campaign as runner
for L in CONFIG['sizes']:
    data = runner.load_pair(OUTPUT_ROOT, L, 'result', runner.identity(CONFIG, L))
    if data is None:
        print(f'L={L}: not yet complete'); continue
    fig, axes = plt.subplots(2, 1, figsize=(3.375, 4.8), layout='constrained')
    axes[0].loglog(data['cycles'][1:], data['total_entropy'][1:]/L, color='#1f77b4')
    axes[0].set(xlabel='cycle t', ylabel='S(t)/Ny')
    image = axes[1].imshow(data['entropy_contour'][-1].T, origin='lower', cmap='magma')
    axes[1].set(xlabel='x', ylabel='y')
    fig.colorbar(image, ax=axes[1], label='s(x,y), final cycle')
    for ax in axes: ax.tick_params(direction='in', top=True, right=True)
    print(f'L={L}, walls={data["walls"].tolist()}, final entropy={data["total_entropy"][-1]:.6g}')
    plt.show()


In [ ]:
from google.colab import runtime
runtime.unassign()
print('done')
